# Omnibus — a year of Line 1, day by day

The `Daten_Linie_1_2024-09_2025-08` window is our **only full year** of data — normally we drop it (it overlaps every event window), but on its own it's perfectly clean, and it's the one source that can show **seasonality**. This notebook lays its 365 days out GitHub-contributions style: one cell per day, colored by that day's median arrival delay, holidays and event days marked.

Caveats: this notebook uses **only** the Line-1 window (so the overlap caveat doesn't apply — there's nothing to overlap with here), `|delay_arr_s| < 7200`, productive arrivals only.

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from datetime import timedelta

df = pl.read_parquet("../data/parquet/features.parquet")
l1 = df.filter((pl.col("source_window") == "Daten_Linie_1_2024-09_2025-08")
               & (pl.col("delay_arr_s").abs() < 7200) & pl.col("productive_arr"))

daily = (l1.group_by("operating_day").agg(
            pl.col("delay_arr_s").median().alias("med"),
            pl.col("delay_arr_s").quantile(0.9).alias("p90"),
            pl.col("is_public_holiday").max().alias("pub"),
            pl.col("is_school_holiday").max().alias("school"),
            pl.col("has_event").max().alias("event"),
            pl.len().alias("n"))
         .sort("operating_day"))
print(f"{daily.height} days, {daily['operating_day'].min()} → {daily['operating_day'].max()}")
daily.head(3)

## Calendar heatmap

Columns are calendar weeks (Mon-aligned), rows are weekdays. Color = median delay that day. Black dots = public holiday, orange ring = event day.

In [ ]:
days = daily["operating_day"].to_list()
med = {d: v for d, v in zip(days, daily["med"].to_list())}
pub = {d for d, b in zip(days, daily["pub"].to_list()) if b}
school = {d for d, b in zip(days, daily["school"].to_list()) if b}
event = {d for d, b in zip(days, daily["event"].to_list()) if b}

start = days[0] - timedelta(days=days[0].weekday())   # back up to Monday
end = days[-1]
ncols = (end - start).days // 7 + 1
grid = np.full((7, ncols), np.nan)
for d, v in med.items():
    delta = (d - start).days
    grid[d.weekday(), delta // 7] = v

fig, ax = plt.subplots(figsize=(18, 4.2))
vmax = float(np.nanpercentile(list(med.values()), 95))
im = ax.imshow(grid, aspect="equal", cmap="RdYlGn_r",
               norm=Normalize(vmin=0, vmax=vmax), origin="upper")

# holiday / event overlays
for d in med:
    col = (d - start).days // 7; row = d.weekday()
    if d in pub:
        ax.plot(col, row, "o", ms=3, color="black")
    if d in event:
        ax.add_patch(plt.Rectangle((col - 0.5, row - 0.5), 1, 1, fill=False,
                                   edgecolor="#ff7f00", linewidth=1.4))

ax.set_yticks(range(7)); ax.set_yticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])
# month labels at the first column of each month
seen = set(); ticks, labels = [], []
for col in range(ncols):
    d0 = start + timedelta(days=col * 7)
    key = (d0.year, d0.month)
    if key not in seen:
        seen.add(key); ticks.append(col); labels.append(d0.strftime("%b\n%Y"))
ax.set_xticks(ticks); ax.set_xticklabels(labels, fontsize=8)
ax.set_title("Line 1 — median arrival delay per day (Sep 2024 → Aug 2025)\n"
             "● public holiday   ▢ event day", fontsize=12)
plt.colorbar(im, ax=ax, fraction=0.015, pad=0.01, label="median delay [s]")
fig.tight_layout(); plt.show()

**How to read it.** Read across a row to see one weekday all year (the Friday row is reliably the reddest); read down a column for a single week. The big patterns to look for: a **summer cool-down** (school out → less traffic, fewer boardings), **Advent/Christmas-market reddening** in December, and whether **public-holiday dots sit on green cells** (light traffic) — if they do, that's evidence holidays *help* Line 1, the inverse of the event effect. Any isolated deep-red cell mid-week is worth clicking through in another notebook (weather? incident?).

## The seasonal signal, distilled

The calendar is the texture; this is the trend line — weekly median delay across the year, with school-holiday weeks shaded. Confirms whether the eye-balled seasonality is real.

In [ ]:
wk = (daily.with_columns(
        week=((pl.col("operating_day") - pl.col("operating_day").min()).dt.total_days() // 7))
      .group_by("week").agg(
            pl.col("med").median().alias("med"),
            pl.col("operating_day").min().alias("wk_start"),
            pl.col("school").mean().alias("school_frac"))
      .sort("week"))

fig, ax = plt.subplots(figsize=(15, 4.5))
ax.plot(wk["wk_start"].to_list(), wk["med"].to_list(), "-o", ms=3, color="#c0392b")
for r in wk.iter_rows(named=True):
    if r["school_frac"] > 0.5:
        ax.axvspan(r["wk_start"], r["wk_start"] + timedelta(days=7),
                   color="#3498db", alpha=0.12)
ax.set_ylabel("weekly median delay [s]"); ax.set_xlabel("week")
ax.set_title("Line 1 weekly median delay — blue bands = school-holiday weeks")
ax.grid(True, alpha=0.25); fig.tight_layout(); plt.show()

corr = daily.select(pl.corr("school", "med")).item()
print(f"school-holiday vs daily median delay correlation: {corr:+.3f}  "
      f"({'school holidays = calmer' if corr < 0 else 'school holidays = busier'})")

**How to read it.** If the red line dips inside the blue (school-holiday) bands, school traffic is a measurable driver of Line 1's delay — a concrete, quantified seasonal lever. The correlation number puts a sign and size on it. This is the kind of year-scale claim only the full Line-1 window can support, and it complements the short event windows that the other notebooks lean on.